# GeoClip Italy Fine-Tuning — Run Guide

This notebook fine-tunes [GeoCLIP](https://github.com/ViktorooReps/geoclip) to predict GPS coordinates (lat/lon) of photos taken in Italy.
It is structured in 7 independent sections so you can jump in at any stage.

---

## Variable dependency map (quick reference)

| Variable | Defined in |
|----------|-----------|
| `MIN_LAT`, `MAX_LAT`, `MIN_LON`, `MAX_LON`, `CHECKPOINT_PATH`, `LOAD_CHECKPOINT`, `BATCH_SIZE`, … | **Global Configuration** (always run first) |
| `LOCAL_METADATA_CSV` (verified) | **Dataset Verification** — overwrites the path from Global Config |
| `val_loader`, `train_loader` | **Dataset & DataLoader** — needs `LOCAL_METADATA_CSV` + images on disk |
| `model`, `device`, `criterion` | **Model Definition & Setup** — needs `CHECKPOINT_PATH` from Global Config |

---

## Which cells to run

### A — Starting from scratch (no dataset, no checkpoint)
Run **everything** top to bottom.

### B — Cleaned dataset already on Drive, training from scratch
| Step | Cell (by title) | Action |
|------|-----------------|--------|
| 1 | Install packages | Run |
| 2 | Global Configuration | Run |
| 3 | Mount Drive | Run |
| 4 | Restore Cleaned Dataset from Drive | Run |
| 5 | Dataset Verification | Run — updates `LOCAL_METADATA_CSV` |
| 6 | Dataset & DataLoader | Run — needs updated `LOCAL_METADATA_CSV` from step 5 |
| 7 | Model Definition & Setup | Run |
| 8 | Training Loop | Run |

### C — Resume training from a checkpoint
Same as B, but in **Global Configuration** set:
```python
LOAD_CHECKPOINT = True
START_EPOCH     = <last completed epoch>
CHECKPOINT_PATH = "<path to your .pth file on Drive>"
```

### D — Single image inference (you have a .pth file, no dataset needed)
| Step | Cell (by title) | Notes |
|------|-----------------|-------|
| 1 | Install packages | — |
| 2 | Global Configuration | Set `LOAD_CHECKPOINT = True`, update `CHECKPOINT_PATH` |
| 3 | Mount Drive | — |
| 4 | Model Definition & Setup | Loads weights; defines `model` and `device` |
| 5 | Single image inference | Set `IMAGE_PATH` to your file |

All other cells → ⏭ SKIP

### E — Full evaluation on validation set (map, km stats, confusion matrix)
Cells 30, 31, 33 all need both `model` **and** `val_loader`.
`val_loader` requires the images on disk and a verified `LOCAL_METADATA_CSV`.

| Step | Cell (by title) | Provides |
|------|-----------------|---------|
| 1 | Install packages | — |
| 2 | Global Configuration | All path/hyperparameter constants |
| 3 | Mount Drive | — |
| 4 | Restore Cleaned Dataset from Drive | Images on disk + `dataset_metadata.csv` |
| 5 | Dataset Verification | Verified `LOCAL_METADATA_CSV` (may rename file) |
| 6 | Dataset & DataLoader | `val_loader` — depends on step 5 output |
| 7 | Model Definition & Setup | `model`, `device` — depends on Drive + Global Config |
| 8 | Map / km test / Full report | Run any of cells 30, 31, 33 |

---
> **Note on PyTorch reinstall (Section 6, optional cell):** only needed if you see CUDA version errors.
> It requires a **Runtime Restart** after execution — do not run it unless necessary.

## 🔧 Section 1: Setup

Run these 3 cells at the start of **every session**, regardless of which path you are taking.

In [1]:
# ============================================================
# INSTALL ALL DEPENDENCIES
# Run this cell first, every session.
# ============================================================

# Core ML stack
!pip install -q geoclip pandas torch torchvision tqdm scikit-learn

# OpenAI CLIP for dataset cleaning
# 1. Remove the wrong 'clip' package if it was previously installed
!pip uninstall -y clip 2>/dev/null || true
# 2. Install the real OpenAI CLIP directly from GitHub
!pip install -q git+https://github.com/openai/CLIP.git
# 3. CLIP runtime dependencies
!pip install -q ftfy regex

# Dataset crawling / HuggingFace
!pip install -q icrawler datasets

# Visualization
!pip install -q folium seaborn matplotlib

print("✅ All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 54.8 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00
✅ All packages installed.


In [2]:
# ============================================================
# GLOBAL CONFIGURATION
# Edit ONLY this cell to adapt the notebook to your Drive layout.
# All other cells reference these variables — never hardcode paths elsewhere.
# ============================================================

import os

# --- GOOGLE DRIVE BASE PATH ---
DRIVE_BASE = "/content/drive/MyDrive/Unipd/Computer Vision Project"

# --- DRIVE SUBDIRECTORIES ---
DRIVE_RAW_DIR        = f"{DRIVE_BASE}/Dataset_GeoClip/Backup_Raw_PreClean_V2"
DRIVE_CLEAN_DIR      = f"{DRIVE_BASE}/Dataset_GeoClip_CLEANED"
DRIVE_CHECKPOINT_DIR = f"{DRIVE_BASE}/Dataset_GeoClip_V5"

# --- LOCAL (Colab runtime) PATHS ---
LOCAL_RAW_DIR      = "dataset_italia_city_enhanced_v2"  # Created by the crawler
LOCAL_IMAGE_DIR    = "dataset_reale"                     # Extracted cleaned images (used by training)
LOCAL_RAW_CSV      = "dataset_train_italia_enhanced_v2.csv"
LOCAL_CLEANED_CSV  = "dataset_italia_CLEANED.csv"
LOCAL_METADATA_CSV = "dataset_metadata.csv"              # Final CSV consumed by training

# --- GEOGRAPHIC BOUNDS (Italy bounding box) ---
MIN_LAT, MAX_LAT = 35.0, 48.0
MIN_LON, MAX_LON =  6.0, 19.0

# --- DATASET CREATION PARAMETERS ---
TARGET_LANDMARKS_COUNT = 3606
IMAGES_PER_LANDMARK    = 20

# --- CLEANING PARAMETERS ---
CLIP_KEEP_THRESHOLD = 0.60   # Images below this CLIP relevance score are deleted
CLIP_BATCH_SIZE     = 32

# --- TRAINING HYPERPARAMETERS ---
BATCH_SIZE           = 8
LEARNING_RATE        = 2e-4
TOTAL_EPOCHS         = 20
CHECKPOINT_FREQUENCY = 2     # Save a full checkpoint every N epochs

# --- CHECKPOINT (for resuming training — see Run Guide above) ---
LOAD_CHECKPOINT = True
START_EPOCH     = 10
CHECKPOINT_PATH = f"{DRIVE_CHECKPOINT_DIR}/checkpoint_epoch_10.pth"
BEST_MODEL_PATH = f"{DRIVE_CHECKPOINT_DIR}/geoclip_italy_BEST.pth"

# Make sure the checkpoint output dir exists
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

print("✅ Configuration loaded.")
print(f"   Drive base      : {DRIVE_BASE}")
print(f"   Checkpoint dir  : {DRIVE_CHECKPOINT_DIR}")
print(f"   Load checkpoint : {LOAD_CHECKPOINT} (from epoch {START_EPOCH})")

✅ Configuration loaded.
   Drive base      : /content/drive/MyDrive/Unipd/Computer Vision Project
   Checkpoint dir  : /content/drive/MyDrive/Unipd/Computer Vision Project/Dataset_GeoClip_V5
   Load checkpoint : True (from epoch 10)


In [4]:
# Mount Google Drive — safe to run multiple times (skips if already mounted)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📥 Section 2: Dataset Creation

Crawls ~{TARGET_LANDMARKS_COUNT} Italian landmarks from the `visheratin/google_landmarks_places` HuggingFace dataset
and downloads up to {IMAGES_PER_LANDMARK} images per landmark via Bing Image Search.

**Skip this section** if you already have the raw dataset on Drive (use "Restore Raw Dataset from Drive" instead).

In [ ]:
# DATASET CREATION V.4 (using visheratin/google_landmarks_places and Bing search)
import csv
import datetime
import logging
import os
import shutil

import pandas as pd
from datasets import load_dataset
from icrawler.builtin import BingImageCrawler
from tqdm.auto import tqdm

# --- Paths from global config ---
OUTPUT_DIR    = LOCAL_RAW_DIR   # "dataset_italia_city_enhanced_v2"
CSV_FILE      = LOCAL_RAW_CSV   # "dataset_train_italia_enhanced_v2.csv"
FAIL_CSV_FILE = "failed_downloads.csv"

failed_landmarks = []

def run_italy_hunter_v2():
    print("🌍 FASE 1: Caricamento Dataset Visheratin...")
    try:
        ds_meta = load_dataset("visheratin/google_landmarks_places", split="train")
        df = ds_meta.to_pandas()
    except Exception as e:
        print(f"❌ Errore dataset: {e}")
        return

    REQUIRED_COLS = {'name', 'lat', 'lon'}
    found_cols = {c.lower() for c in df.columns}

    if not REQUIRED_COLS.issubset(found_cols):
        missing = REQUIRED_COLS - found_cols
        raise ValueError(f"❌ Colonne mancanti: {missing}")

    # Auto-detect delle colonne
    cols = list(df.columns)
    # Cerchiamo le colonne chiave in modo flessibile
    c_name = next((c for c in cols if 'name' in c.lower()), 'name')
    c_lat  = next((c for c in cols if 'lat'  in c.lower()), 'latitude')
    c_lon  = next((c for c in cols if 'lon'  in c.lower()), 'longitude')
    c_city = next((c for c in cols if 'city' in c.lower()), None)  # <--- ECCOLA

    print(f"   ✅ Colonne rilevate: Name='{c_name}', City='{c_city}'")

    # --- FASE 2: FILTRO ITALIA (Geografico) ---
    print("\n🔍 FASE 2: Filtraggio Area Italia...")

    # Usiamo il rettangolo geografico per sicurezza
    geo_mask = (
        (df[c_lat] >= 36.6) & (df[c_lat] <= 47.1) &
        (df[c_lon] >= 6.6)  & (df[c_lon] <= 18.5)
    )
    df_italy = df[geo_mask].copy()
    print(f"   🇮🇹 Monumenti in area italiana: {len(df_italy)}")

    # Campionamento Casuale
    if len(df_italy) > TARGET_LANDMARKS_COUNT:
        print(f"   🎲 Seleziono {TARGET_LANDMARKS_COUNT} monumenti casuali...")
        df_target = df_italy.sample(n=TARGET_LANDMARKS_COUNT, random_state=42)
    else:
        df_target = df_italy

    # --- FASE 3: DOWNLOAD CON PARAMETRO CITTÀ ---
    print(f"\n🚀 FASE 3: Avvio Download (Query = Nome + Città + Italy)...")

    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR)

    # Inizializza CSV
    with open(CSV_FILE, 'w', newline='') as f:
        # Aggiungo anche la colonna 'city' nel CSV finale per tua comodità
        csv.writer(f).writerow(['image_path', 'latitude', 'longitude', 'landmark_name', 'city'])

    count = 0
    success_landmarks = 0

    # Disattiva log rumorosi di icrawler
    logging.getLogger("icrawler").setLevel(logging.ERROR)

    pbar = tqdm(df_target.iterrows(), total=len(df_target), desc="Crawling")

    for _, row in pbar:
        name = str(row[c_name]).strip()
        lat  = row[c_lat]
        lon  = row[c_lon]

        # Gestione della Città
        city_val = ""
        if c_city and pd.notna(row[c_city]):
            city_val = str(row[c_city]).strip()

        # --- LA NUOVA QUERY POTENZIATA ---
        # Se la città c'è, la usiamo. Altrimenti usiamo solo il nome.
        if city_val:
            query = f"{name} {city_val} Italy landmark"
        else:
            query = f"{name} Italy landmark"

        # Prepariamo la cartella
        # Nome cartella pulito: "Roma_Colosseo" oppure solo "Colosseo"
        folder_prefix = city_val.replace(" ", "") + "_" if city_val else ""
        clean_name    = "".join([c if c.isalnum() else "_" for c in name])[:20]
        folder_name   = f"{folder_prefix}{clean_name}"

        save_dir = os.path.join(OUTPUT_DIR, folder_name)
        os.makedirs(save_dir, exist_ok=True)

        try:
            crawler = BingImageCrawler(storage={'root_dir': save_dir}, downloader_threads=4)
            crawler.crawl(keyword=query, max_num=IMAGES_PER_LANDMARK, file_idx_offset='auto')

            # Post-Processing
            files = [f for f in os.listdir(save_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

            if not files:
                os.rmdir(save_dir)
                continue

            for fname in files:
                full_path = os.path.join(save_dir, fname)
                with open(CSV_FILE, 'a', newline='') as f:
                    # Salviamo anche la città nel CSV
                    csv.writer(f).writerow([full_path, lat, lon, name, city_val])
                count += 1

            success_landmarks += 1
            # Aggiorna la barra mostrando la query attuale
            pbar.set_description(f"📸 {success_landmarks} ok | Q: {query[:30]}...")

        except Exception as e:
            failed_landmarks.append({'name': name, 'error': str(e)})
            continue

    print(f"\n🏆 FINE! Dataset creato.")
    print(f"   Immagini totali: {count}")
    print(f"   CSV salvato: {CSV_FILE}")


run_italy_hunter_v2()
if failed_landmarks:
    pd.DataFrame(failed_landmarks).to_csv(FAIL_CSV_FILE, index=False)
    print(f"⚠️ {len(failed_landmarks)} download falliti salvati in: {FAIL_CSV_FILE}")

## 💾 Section 3: Drive Backup and Restore

Use these cells to persist and recover the dataset between Colab sessions.
Run only the cell(s) relevant to your current step.

| Cell | When to run |
|------|-------------|
| **Backup Raw** | After dataset creation, before cleaning |
| **Restore Raw** | To re-run cleaning on a previously downloaded dataset |
| **Backup Cleaned** | After CLIP cleaning, before training |
| **Restore Cleaned** | ✅ Most common — to start/resume training with an existing dataset |

### Backup Raw Dataset to Drive

In [ ]:
# SAVING RAW DATASET TO DRIVE
import datetime
import os
import shutil

# --- Paths from global config ---
LOCAL_IMAGE_DIR_ = LOCAL_RAW_DIR   # "dataset_italia_city_enhanced_v2"
LOCAL_CSV_FILE_  = LOCAL_RAW_CSV   # "dataset_train_italia_enhanced_v2.csv"
BACKUP_DIR       = DRIVE_RAW_DIR   # set in global config

if not os.path.exists(LOCAL_IMAGE_DIR_):
    print(f"❌ ERRORE: Non trovo la cartella immagini '{LOCAL_IMAGE_DIR_}'.")
    print("   Hai eseguito lo script di download prima?")
else:
    os.makedirs(BACKUP_DIR, exist_ok=True)

    # 1. Zippiamo le immagini
    print(f"📦 Compressione cartella '{LOCAL_IMAGE_DIR_}'...")
    # Aggiungiamo un timestamp per sicurezza
    timestamp    = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    zip_filename = f"dataset_raw_images_{timestamp}"
    shutil.make_archive(zip_filename, 'zip', LOCAL_IMAGE_DIR_)

    # 2. Copia su Drive
    print(f"🚀 Caricamento su Drive in: {BACKUP_DIR}")

    # Copia ZIP
    shutil.copy(f"{zip_filename}.zip", os.path.join(BACKUP_DIR, "images_raw.zip"))
    print("   ✅ ZIP Immagini copiato.")

    # Copia CSV metadata
    if os.path.exists(LOCAL_CSV_FILE_):
        shutil.copy(LOCAL_CSV_FILE_, os.path.join(BACKUP_DIR, "metadata_raw.csv"))
        print("   ✅ CSV Metadata copiato.")
    else:
        print("   ⚠️ CSV non trovato!")

    # Copia CSV dei download falliti (se esiste)
    if os.path.exists("failed_downloads.csv"):
        shutil.copy("failed_downloads.csv", os.path.join(BACKUP_DIR, "failed_downloads.csv"))
        print("   ✅ CSV Fail copiato.")

    print("\n🏆 BACKUP COMPLETATO!")
    print("   Ora puoi procedere con la pulizia (CLIP) senza paura.")

### Restore Raw Dataset from Drive

In [6]:
# LOADING RAW DATASET FROM DRIVE
import os
import shutil

# --- Paths from global config ---
DRIVE_SOURCE_DIR = DRIVE_RAW_DIR   # set in global config
LOCAL_IMAGE_DIR_ = "dataset_italia_city_enhanced"  # expected by the cleaning script
LOCAL_CSV_NAME_  = "dataset_train_italia_enhanced.csv"

# 1. PULIZIA AMBIENTE (Per essere sicuri di sovrascrivere tutto)
print("🧹 Pulizia ambiente locale (rimozione vecchi dati)...")
if os.path.exists(LOCAL_IMAGE_DIR_):
    shutil.rmtree(LOCAL_IMAGE_DIR_)  # Cancella cartella immagini
if os.path.exists(LOCAL_CSV_NAME_):
    os.remove(LOCAL_CSV_NAME_)       # Cancella CSV
if os.path.exists("temp_images.zip"):
    os.remove("temp_images.zip")

# 2. COPIA DA DRIVE
print(f"🚀 Recupero dati da: {DRIVE_SOURCE_DIR}")
src_zip = os.path.join(DRIVE_SOURCE_DIR, "images_raw.zip")
src_csv = os.path.join(DRIVE_SOURCE_DIR, "metadata_raw.csv")

if not os.path.exists(src_zip):
    print(f"❌ ERRORE: Non trovo il file {src_zip}")
    print("   Controlla che il backup esista su Drive.")
else:
    # Copia CSV e lo rinomina come serve a noi
    shutil.copy(src_csv, LOCAL_CSV_NAME_)
    print("   ✅ CSV Metadata recuperato.")

    # Copia ZIP
    shutil.copy(src_zip, "temp_images.zip")
    print("   ✅ ZIP Immagini copiato.")

    # 3. ESTRAZIONE
    print(f"📦 Estrazione in corso nella cartella '{LOCAL_IMAGE_DIR_}'...")
    os.makedirs(LOCAL_IMAGE_DIR_, exist_ok=True)
    # Usiamo unzip -d per dire "metti i file QUI DENTRO"
    os.system(f'unzip -q temp_images.zip -d "{LOCAL_IMAGE_DIR_}"')
    os.remove("temp_images.zip")

    print("\n🏆 RIPRISTINO COMPLETATO!")
    print(f"   📂 Cartella pronta: {LOCAL_IMAGE_DIR_}")
    print(f"   📄 File CSV pronto: {LOCAL_CSV_NAME_}")
    print("   👉 Ora puoi lanciare lo script di pulizia CLIP.")

🧹 Pulizia ambiente locale (rimozione vecchi dati)...
🚀 Recupero dati da: /content/drive/MyDrive/Unipd/Computer Vision Project/Dataset_GeoClip/Backup_Raw_PreClean_V2
   ✅ CSV Metadata recuperato.


KeyboardInterrupt: 

### Backup Cleaned Dataset to Drive

In [ ]:
# SAVING CLEANED DATASET TO DRIVE
import os
import shutil

# --- Paths from global config ---
DRIVE_DEST   = DRIVE_CLEAN_DIR   # set in global config
LOCAL_IMAGES = "dataset_italia_city_enhanced"
LOCAL_CSV_   = LOCAL_CLEANED_CSV  # "dataset_italia_CLEANED.csv"

os.makedirs(DRIVE_DEST, exist_ok=True)
print(f"📂 Destinazione Drive: {DRIVE_DEST}")

# 1. Creazione dello ZIP (della cartella locale delle immagini)
print("📦 Compressione del dataset in corso...")
# Crea un file temporaneo 'dataset_backup.zip' nell'ambiente Colab
shutil.make_archive("dataset_backup", 'zip', LOCAL_IMAGES)

# 2. Copia su Drive
print("🚀 Caricamento file su Drive...")

zip_dest = os.path.join(DRIVE_DEST, "dataset_images.zip")
csv_dest = os.path.join(DRIVE_DEST, "dataset_metadata.csv")

# Copia lo ZIP
shutil.copy("dataset_backup.zip", zip_dest)
print(f"   ✅ ZIP salvato in: {zip_dest}")

# Copia il CSV Pulito (assumendo tu abbia fatto lo step di pulizia CSV prima)
# Se il file si chiama diversamente, cambia LOCAL_CLEANED_CSV nella config globale
if os.path.exists(LOCAL_CSV_):
    shutil.copy(LOCAL_CSV_, csv_dest)
    print(f"   ✅ CSV salvato in: {csv_dest}")
else:
    print("   ⚠️ Attenzione: File CSV pulito non trovato. Hai eseguito lo script di aggiornamento CSV?")

print("\n🏆 SALVATAGGIO COMPLETATO SU DRIVE!")

### Restore Cleaned Dataset from Drive

✅ **Most common restore path** — run this to start or resume training.

In [5]:
# COPY CLEANED DATASET FROM DRIVE
import os
import shutil

# --- Paths from global config ---
DRIVE_SOURCE  = DRIVE_CLEAN_DIR   # set in global config
ZIP_NAME      = "dataset_images.zip"
CSV_NAME      = "dataset_coordinate_FINGERPRINT.csv"  # actual filename saved on Drive by V2
LOCAL_EXTRACT = LOCAL_IMAGE_DIR   # "dataset_reale"

print(f"🔄 Cerco i file in: {DRIVE_SOURCE}")

src_zip = os.path.join(DRIVE_SOURCE, ZIP_NAME)
src_csv = os.path.join(DRIVE_SOURCE, CSV_NAME)

# Controllo di sicurezza
if not os.path.exists(src_zip):
    print(f"❌ ERRORE: Non trovo il file {src_zip}")
    print("   Controlla di aver scritto bene il percorso e che il file esista su Drive.")
else:
    # 1. Copia i file da Drive a Colab (nella cartella corrente ".")
    print("🚀 Copia in corso da Drive al Runtime (veloce)...")
    shutil.copy(src_zip, ".")
    shutil.copy(src_csv, LOCAL_METADATA_CSV)  # rinominato per il training
    print("   ✅ File copiati.")

    # 2. Estrazione ZIP
    print(f"📦 Estrazione immagini in '{LOCAL_EXTRACT}'...")
    # Usiamo un comando di sistema per scompattare (è più veloce di Python per tanti file)
    # Nota le virgolette extra f'"{ZIP_NAME}"' per sicurezza sugli spazi, anche se qui non ce ne sono
    os.system(f'unzip -q "{ZIP_NAME}" -d "{LOCAL_EXTRACT}"')

    print("\n🏆 TUTTO PRONTO!")
    print(f"   📂 Cartella Immagini: {LOCAL_EXTRACT}")
    print(f"   📄 File CSV Metadata: {LOCAL_METADATA_CSV}")

    # Piccola verifica
    num_files = sum(len(files) for _, _, files in os.walk(LOCAL_EXTRACT))
    print(f"   📊 File recuperati: {num_files}")

🔄 Cerco i file in: /content/drive/MyDrive/Unipd/Computer Vision Project/Dataset_GeoClip_CLEANED
🚀 Copia in corso da Drive al Runtime (veloce)...
   ✅ File copiati.
📦 Estrazione immagini in 'dataset_reale'...

🏆 TUTTO PRONTO!
   📂 Cartella Immagini: dataset_reale
   📄 File CSV Metadata: dataset_metadata.csv
   📊 File recuperati: 39105


## 🧹 Section 4: Dataset Cleaning

Uses OpenAI CLIP (`ViT-B/32`) to score every image against the prompt
*"a photo of an outdoor historical landmark or building"*.
Images scoring below `CLIP_KEEP_THRESHOLD` (default 0.60) are deleted.

**Skip this section** if you are loading a pre-cleaned dataset from Drive.

In [ ]:
# CLIP-based image quality filtering
import os

import clip
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# --- Paths/params from global config ---
DATASET_DIR     = "dataset_italia_city_enhanced"
KEEP_THRESHOLD  = CLIP_KEEP_THRESHOLD   # 0.60
BATCH_SIZE_CLIP = CLIP_BATCH_SIZE       # 32

# 1. Caricamento CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Caricamento modello CLIP su {device}...")
model, preprocess = clip.load("ViT-B/32", device=device)

text_inputs = torch.cat([clip.tokenize(c) for c in [
    "a photo of an outdoor historical landmark or building",
    "a selfie or a portrait of a person",
    "a photo of food or drink",
    "an indoor view of a room or hotel",
    "a map or a screenshot"
]]).to(device)

# 2. PREPARAZIONE LISTA FILE
print("📂 Indicizzazione file in corso...")
all_images = []
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append(os.path.join(root, file))

print(f"🧹 INIZIO PULIZIA SU {len(all_images)} IMMAGINI...")

removed_count = 0
total_count   = len(all_images)
failed_clean  = []  # ✅ SPOSTATO FUORI dal loop!

# 3. CICLO A BATCH
pbar = tqdm(range(0, len(all_images), BATCH_SIZE_CLIP), desc="Cleaning")

for i in pbar:
    batch_paths  = all_images[i:i + BATCH_SIZE_CLIP]
    batch_images = []
    valid_paths  = []  # ✅ Per tracciare quali immagini sono valide

    # Carica il batch
    for file_path in batch_paths:
        try:
            img = preprocess(Image.open(file_path))
            batch_images.append(img)
            valid_paths.append(file_path)
        except Exception as e:
            tqdm.write(f"⚠️ Errore su {file_path}: {e}")
            failed_clean.append({'file': file_path, 'error': str(e)})

    if not batch_images:
        continue

    batch_tensor = torch.stack(batch_images).to(device)  # ✅ stack invece di cat

    with torch.no_grad():
        image_features = model.encode_image(batch_tensor)
        text_features  = model.encode_text(text_inputs)

        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features  /= text_features.norm(dim=-1, keepdim=True)
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)

        is_landmark_scores = similarity[:, 0]  # ✅ Tutti gli score, non solo il primo!

    # Rimuovi le immagini sotto threshold
    for file_path, score in zip(valid_paths, is_landmark_scores):
        if score.item() < KEEP_THRESHOLD:
            os.remove(file_path)
            removed_count += 1

    pbar.set_postfix({"Rimosse": removed_count})

# Salva errori se ce ne sono
if failed_clean:
    pd.DataFrame(failed_clean).to_csv('failed_clean.csv', index=False)
    print(f"⚠️ {len(failed_clean)} immagini con errori salvate in failed_clean.csv")

print(f"\n✨ PULIZIA COMPLETATA!")
print(f"   Immagini iniziali: {total_count}")
print(f"   Immagini rimosse:  {removed_count}")
print(f"   Immagini rimaste:  {total_count - removed_count}")

In [ ]:
# UPDATE CSV POST CLEANING
# FILTRO MAGICO: Tieni la riga SOLO se il file esiste ancora sul disco
# (os.path.exists controlla se il file c'è davvero)
import os

import pandas as pd

# --- Paths from global config ---
OLD_CSV    = "dataset_train_italia_enhanced.csv"
CLEAN_CSV  = LOCAL_CLEANED_CSV   # "dataset_italia_CLEANED.csv"
DATASET_DIR = "dataset_italia_city_enhanced"  # La cartella delle immagini

print("🔄 Aggiornamento del CSV post-pulizia...")

if os.path.exists(OLD_CSV):
    df           = pd.read_csv(OLD_CSV)
    original_len = len(df)

    df_clean = df[df['image_path'].apply(os.path.exists)]

    df_clean.to_csv(CLEAN_CSV, index=False)

    print(f"✅ CSV Aggiornato!")
    print(f"   Righe originali:      {original_len}")
    print(f"   Righe valide rimaste: {len(df_clean)}")
    print(f"   File salvato come:    {CLEAN_CSV}")
else:
    print("❌ Attenzione: Non trovo il file CSV originale.")

## ✅ Section 5: Dataset Verification

Checks that every row in the CSV has a corresponding image on disk.
Rows pointing to missing files are dropped and a verified CSV is saved.
Always run this before training.

In [6]:
# NUOVA CELLA - Inserisci PRIMA della cella 15
# Verifica che tutte le immagini nel CSV esistano
import os

import pandas as pd


def verify_dataset_integrity(csv_path, image_dir):
    df = pd.read_csv(csv_path)

    print(f"🔍 Verifica integrità dataset...")
    print(f"   Righe nel CSV: {len(df)}")

    missing = []
    for idx, row in df.iterrows():
        img_path = os.path.join(image_dir, row['path'])
        if not os.path.exists(img_path):
            missing.append(idx)

    if missing:
        print(f"❌ {len(missing)} immagini mancanti!")
        print(f"   Prime 5: {missing[:5]}")

        # Crea versione pulita
        df_clean  = df.drop(missing)
        clean_csv = csv_path.replace('.csv', '_verified.csv')
        df_clean.to_csv(clean_csv, index=False)
        print(f"   ✅ CSV pulito salvato: {clean_csv}")
        return clean_csv
    else:
        print(f"✅ Tutte le {len(df)} immagini sono presenti!")
        return csv_path


# --- Paths from global config ---
LOCAL_METADATA_CSV = verify_dataset_integrity(LOCAL_METADATA_CSV, LOCAL_IMAGE_DIR)

🔍 Verifica integrità dataset...
   Righe nel CSV: 0
✅ Tutte le 0 immagini sono presenti!


## 🚀 Section 6: Training

Fine-tunes GeoCLIP for Italian geolocation.

**Cell order:**
1. *(Optional)* Reinstall PyTorch — only if you see CUDA version errors
2. Dataset & DataLoader — always run
3. Clear old model from GPU memory — run if restarting training in the same session
4. Model Definition & Training Setup — always run
5. Training Loop — always run
6. *(Optional)* Training Diagnostics — run to inspect gradient flow after a few steps

### (Optional) Reinstall PyTorch for CUDA

> ⚠️ **Only run this cell if you see CUDA version mismatch errors.**
> After running it you **must restart the runtime** (Runtime → Restart runtime) before continuing.

In [ ]:
# Nuova cella all'inizio del notebook
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Riavvia runtime dopo questo
print("✅ PyTorch reinstallato. RIAVVIA IL RUNTIME (Runtime → Restart runtime)")

In [7]:
# Dataset & DataLoader
import os

import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm


class ItalyGeoDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row['path']

        # NORMALIZZAZIONE: Trasformiamo le coordinate reali in un numero tra 0 e 1
        # Esempio: Se Lat è 35 (Min) diventa 0. Se è 48 (Max) diventa 1.
        lat, lon = row['lat'], row['lon']

        # Validazione
        if pd.isna(lat) or pd.isna(lon):
            raise ValueError(f"Coordinate invalide in riga {idx}")

        # Clipping per sicurezza
        lat = max(MIN_LAT, min(MAX_LAT, lat))
        lon = max(MIN_LON, min(MAX_LON, lon))

        norm_lat = (lat - MIN_LAT) / (MAX_LAT - MIN_LAT)
        norm_lon = (lon - MIN_LON) / (MAX_LON - MIN_LON)

        target = torch.tensor([norm_lat, norm_lon], dtype=torch.float32)

        try:
            image = Image.open(os.path.join(LOCAL_IMAGE_DIR, img_path)).convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224), color='black')

        if self.transform:
            image = self.transform(image)

        return image, target


# Trasformazioni (Standard GeoClip)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # No augmentation per validation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

# Caricamento Dati
full_df = pd.read_csv(os.path.join(DRIVE_CLEAN_DIR, "dataset_coordinate_FINGERPRINT.csv"))

# Normalize column names — handles CSV files from different pipeline versions:
# dataset creation writes 'image_path'/'latitude'/'longitude',
# some older checkpoints used 'path'/'lat'/'lon'.
col_map = {}
for c in full_df.columns:
    if c.lower() == 'latitude':     col_map[c] = 'lat'
    elif c.lower() == 'longitude':  col_map[c] = 'lon'
    elif c.lower() == 'image_path': col_map[c] = 'path'
if col_map:
    full_df = full_df.rename(columns=col_map)

# Filtriamo eventuali dati sporchi fuori dall'Italia (sicurezza)
full_df = full_df[
    (full_df['lat'] >= MIN_LAT) & (full_df['lat'] <= MAX_LAT) &  # Changed from 'latitude' to 'lat'
    (full_df['lon'] >= MIN_LON) & (full_df['lon'] <= MAX_LON)    # Changed from 'longitude' to 'lon'
]

train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=42)

train_loader = DataLoader(
    ItalyGeoDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    ItalyGeoDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"✅ Dataset caricato: {len(train_df)} train | {len(val_df)} val")

✅ Dataset caricato: 28891 train | 7223 val


In [10]:
# Clear old model from GPU memory before re-initialising
# Run this if you are restarting training in the same session
try:
    del model
    print("🗑️ Vecchio modello eliminato dalla memoria.")
except NameError:
    print("Mmm, non c'era nessun modello.")

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Mmm, non c'era nessun modello.


In [8]:
# Model Definition & Training Setup
import copy
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim
from geoclip.model import GeoCLIP
from tqdm import tqdm

BEST_MODEL_PATH
# ==========================================
# 1. DEFINIZIONE CLASSE "MANUALE"
# ==========================================
class GeoClipItaly(nn.Module):
    def __init__(self):
        super(GeoClipItaly, self).__init__()
        print("🏗️ Inizializzazione GeoClipItaly (Fine-tuning parziale)...")
        self.geoclip = GeoCLIP(from_pretrained=True)

        # 1️⃣ Freeze TUTTO
        for param in self.geoclip.parameters():
            param.requires_grad = False

        # 2️⃣ Sblocca ULTIMI 2 LAYER del Vision Transformer
        num_layers = len(self.geoclip.image_encoder.CLIP.vision_model.encoder.layers)
        print(f"   📊 Vision Transformer ha {num_layers} layer")

        # Sblocca layer 22 e 23 (ultimi 2 su 24)
        for i in range(num_layers - 2, num_layers):
            print(f"   🔓 Sbloccando layer {i}")
            for param in self.geoclip.image_encoder.CLIP.vision_model.encoder.layers[i].parameters():
                param.requires_grad = True

        self.geoclip.eval()

        # Regressor uguale
        input_dim = 512
        self.regressor = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Linear(128, 2),
            nn.Sigmoid()
        )

        # Verifica
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   ✅ Parametri trainable totali: {trainable:,}")

    def train(self, mode=True):
        # Override train() per tenere GeoCLIP sempre in eval
        super().train(mode)
        self.geoclip.eval()  # GeoCLIP sempre frozen
        return self

    def forward(self, x):
        vision_model  = self.geoclip.image_encoder.CLIP.vision_model
        vision_output = vision_model(pixel_values=x)

        if hasattr(vision_output, 'pooler_output') and vision_output.pooler_output is not None:
            pooled = vision_output.pooler_output
        else:
            pooled = vision_output.last_hidden_state[:, 0, :]

        projected    = self.geoclip.image_encoder.CLIP.visual_projection(pooled)
        features_512 = self.geoclip.image_encoder.mlp(projected)

        return self.regressor(features_512)


# ==========================================
# FUNZIONE METRICA
# ==========================================
def haversine_distance_batch(pred, target):
    # Calcola distanza Haversine in km
    pred_lat   = pred[:, 0]   * (MAX_LAT - MIN_LAT) + MIN_LAT
    pred_lon   = pred[:, 1]   * (MAX_LON - MIN_LON) + MIN_LON
    target_lat = target[:, 0] * (MAX_LAT - MIN_LAT) + MIN_LAT
    target_lon = target[:, 1] * (MAX_LON - MIN_LON) + MIN_LON

    R    = 6371
    dlat = torch.deg2rad(target_lat - pred_lat)
    dlon = torch.deg2rad(target_lon - pred_lon)

    a = (torch.sin(dlat / 2) ** 2
         + torch.cos(torch.deg2rad(pred_lat))
         * torch.cos(torch.deg2rad(target_lat))
         * torch.sin(dlon / 2) ** 2)
    c = 2 * torch.atan2(torch.sqrt(a), torch.sqrt(1 - a))

    return R * c


# ✅ LOSS COMBINATA: L1 + penalità distanza geografica
def combined_loss(pred, target):
    # L1 loss sulle coordinate normalizzate
    l1 = nn.functional.l1_loss(pred, target)

    # Haversine loss (distanza in km)
    distances = haversine_distance_batch(pred, target)
    # Normalizza in range simile a L1 (dividi per 1000 km)
    haversine = distances.mean() / 1000.0

    # Combina: 5% L1, 95% Haversine
    return 0.05 * l1 + 0.95 * haversine


# ==========================================
# 2. SETUP E AVVIO
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Creazione
model = GeoClipItaly().to(device)

# ✅ CARICAMENTO CHECKPOINT COMPLETO
if LOAD_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 Caricamento checkpoint da: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    # Se il checkpoint è vecchio (solo state_dict)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=TOTAL_EPOCHS,  # Scende fino a epoch TOTAL_EPOCHS
            eta_min=1e-6         # LR minimo
        )
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    else:
        # Checkpoint vecchio formato
        model.load_state_dict(checkpoint, strict=False)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=TOTAL_EPOCHS,
            eta_min=1e-6
        )
        best_val_loss = float('inf')

    print("✅ Checkpoint caricato.")
else:
    if LOAD_CHECKPOINT:
        print(f"⚠️ Checkpoint non trovato in: {CHECKPOINT_PATH}")
        print("   Avvio training da zero.")
    best_val_loss = float('inf')
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TOTAL_EPOCHS,
        eta_min=1e-6
    )

criterion = combined_loss

🏗️ Inizializzazione GeoClipItaly (Fine-tuning parziale)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

   📊 Vision Transformer ha 24 layer
   🔓 Sbloccando layer 22
   🔓 Sbloccando layer 23
   ✅ Parametri trainable totali: 26,311,810
🔄 Caricamento checkpoint da: /content/drive/MyDrive/Unipd/Computer Vision Project/Dataset_GeoClip_V5/checkpoint_epoch_10.pth
✅ Checkpoint caricato.


In [ ]:
# ==========================================
# 3. TRAINING LOOP
# ==========================================
print(f"🔥 INIZIO TRAINING: Epoche {START_EPOCH+1} -> {TOTAL_EPOCHS} 🔥")

for epoch in range(START_EPOCH, TOTAL_EPOCHS):
    model.train()
    train_loss      = 0.0
    train_distances = []

    loop = tqdm(train_loader, desc=f"Epoca {epoch+1}/{TOTAL_EPOCHS} [Train]", leave=False)

    for imgs, targets in loop:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = model(imgs)
        loss    = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

        # ✅ Calcola distanze
        with torch.no_grad():
            distances = haversine_distance_batch(outputs, targets)
            train_distances.extend(distances.cpu().numpy())

        loop.set_postfix(loss=loss.item(), dist_km=distances.mean().item())

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_train_dist = sum(train_distances) / len(train_distances)

    # Validazione
    model.eval()
    val_loss      = 0.0
    val_distances = []

    val_loop = tqdm(val_loader, desc=f"Epoca {epoch+1}/{TOTAL_EPOCHS} [Val  ]", leave=False)
    with torch.no_grad():
        for imgs, targets in val_loop:
            imgs, targets = imgs.to(device), targets.to(device)
            outputs = model(imgs)

            #loss = criterion(outputs, targets)
            if callable(criterion):
                loss = criterion(outputs, targets)
            else:
                loss = nn.functional.l1_loss(outputs, targets)

            val_loss += loss.item() * imgs.size(0)

            # ✅ Calcola distanze
            distances = haversine_distance_batch(outputs, targets)
            val_distances.extend(distances.cpu().numpy())

            val_loop.set_postfix(loss=loss.item(), dist_km=distances.mean().item())

    avg_val_loss = val_loss / len(val_loader.dataset)
    avg_val_dist = sum(val_distances) / len(val_distances)

    #scheduler.step(avg_val_loss)
    scheduler.step()

    # ✅ STAMPA CON METRICHE GEOGRAFICHE
    print(f"✅ Epoca {epoch+1}/{TOTAL_EPOCHS}:")
    print(f"   Train - Loss: {avg_train_loss:.5f} | Dist: {avg_train_dist:.2f} km")
    print(f"   Val   - Loss: {avg_val_loss:.5f} | Dist: {avg_val_dist:.2f} km")

    # ✅ SALVA BEST MODEL
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"   🏆 Nuovo miglior modello! Val Loss: {avg_val_loss:.5f}")

    # ✅ CHECKPOINT COMPLETO
    if (epoch + 1) % CHECKPOINT_FREQUENCY == 0:
        ckpt_name  = f"checkpoint_epoch_{epoch+1}.pth"
        checkpoint = {
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss':           avg_train_loss,
            'val_loss':             avg_val_loss,
            'best_val_loss':        best_val_loss,
            'train_dist':           avg_train_dist,
            'val_dist':             avg_val_dist,
        }
        torch.save(checkpoint, os.path.join(DRIVE_CHECKPOINT_DIR, ckpt_name))
        print(f"   💾 Checkpoint completo salvato.")

print("\n🎉 TRAINING COMPLETATO!")
print(f"Best Val Loss: {best_val_loss:.5f}")

In [ ]:
# ===== CELLA DIAGNOSTICA (FIXED) =====
# Run after a few training steps to inspect gradient flow and learning rate.
import matplotlib.pyplot as plt

print("🔍 DIAGNOSI TRAINING")
print("=" * 50)

# 1. Check Learning Rate
print(f"\n📊 Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
if optimizer.param_groups[0]['lr'] < 1e-5:
    print("⚠️ WARNING: LR molto basso! Potrebbe essere troppo lento.")

# 2. Check Gradient Flow
model.train()
sample_batch        = next(iter(train_loader))
imgs, targets       = sample_batch[0].to(device), sample_batch[1].to(device)

optimizer.zero_grad()
outputs = model(imgs)
loss    = criterion(outputs, targets)
loss.backward()

# Controlla i gradienti
grad_norms = []
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_norm = param.grad.norm().item()
        grad_norms.append((name, grad_norm))
        if grad_norm < 1e-7:
            print(f"⚠️ Gradiente quasi-zero in: {name} ({grad_norm:.2e})")

if not grad_norms:
    print("❌ NESSUN GRADIENTE TROVATO! Il modello potrebbe essere completamente frozen.")
else:
    print(f"\n✅ Trovati {len(grad_norms)} layer con gradienti attivi.")
    avg_grad = sum(g for _, g in grad_norms) / len(grad_norms)
    print(f"   Norma media dei gradienti: {avg_grad:.4f}")

# 3. Check Output Range
model.eval()
with torch.no_grad():
    sample_out = model(imgs)
    print(f"\n📤 Output range - Min: {sample_out.min():.4f}, Max: {sample_out.max():.4f}")
    print(f"   (Atteso: tra 0 e 1, visto che usiamo Sigmoid in output)")

# 4. Verifica Loss
print(f"\n📉 Loss sul batch di test: {loss.item():.5f}")

optimizer.zero_grad()  # Pulizia gradienti di test

## 📊 Section 7: Testing and Evaluation

Visualise predictions on an interactive map, compute km-error statistics,
and generate a full classification report (Nord/Centro/Sud macro-regions).

In [13]:
# --- SCRIPT DI VISUALIZZAZIONE MAPPA ---
import folium  # Libreria per le mappe interattive
import torch
from IPython.display import display


def mappa_risultati(model, loader, device, num_samples=5):
    model.eval()

    # Prepara la mappa centrata sull'Italia
    m = folium.Map(location=[42.5, 12.5], zoom_start=6)

    iterator        = iter(loader)
    images, targets = next(iterator)
    images          = images.to(device)

    with torch.no_grad():
        preds   = model(images).cpu().numpy()
        targets = targets.numpy()

    print(f"📍 Generazione mappa per {num_samples} monumenti...")

    for i in range(min(num_samples, len(preds))):
        # Denormalizzazione
        p_lat = preds[i][0]   * (MAX_LAT - MIN_LAT) + MIN_LAT
        p_lon = preds[i][1]   * (MAX_LON - MIN_LON) + MIN_LON
        t_lat = targets[i][0] * (MAX_LAT - MIN_LAT) + MIN_LAT
        t_lon = targets[i][1] * (MAX_LON - MIN_LON) + MIN_LON

        # Linea che collega la predizione alla realtà
        folium.PolyLine([(t_lat, t_lon), (p_lat, p_lon)], color="red", weight=2.5, opacity=0.8).add_to(m)

        # Marker VERO (Verde)
        folium.Marker(
            [t_lat, t_lon],
            popup=f"Target Reale",
            icon=folium.Icon(color="green", icon="flag")
        ).add_to(m)

        # Marker PREDETTO (Rosso)
        folium.Marker(
            [p_lat, p_lon],
            popup=f"Predizione AI",
            icon=folium.Icon(color="red", icon="robot")
        ).add_to(m)

    display(m)


# Lancia la mappa (assicurati di aver finito il training prima!)
mappa_risultati(model, val_loader, device)

In [10]:
# --- TEST DI VERIFICA IMMEDIATO ---
# Dobbiamo "denormalizzare" per vedere i KM reali
import numpy as np
import torch
from math import asin, cos, radians, sin, sqrt


def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return c * 6371


print("\n🌍 TEST ERRORI REALI (KM):")
model.eval()
errors   = []
iterator = iter(val_loader)

for _ in range(5):  # Testa sui primi 5 batch
    try:
        imgs, targets = next(iterator)
    except StopIteration:
        break

    imgs = imgs.to(device)
    with torch.no_grad():
        preds = model(imgs).cpu().numpy()

    targets = targets.numpy()

    for i in range(len(preds)):
        pred_lat = preds[i][0]   * (MAX_LAT - MIN_LAT) + MIN_LAT
        pred_lon = preds[i][1]   * (MAX_LON - MIN_LON) + MIN_LON
        true_lat = targets[i][0] * (MAX_LAT - MIN_LAT) + MIN_LAT
        true_lon = targets[i][1] * (MAX_LON - MIN_LON) + MIN_LON

        dist = haversine(pred_lon, pred_lat, true_lon, true_lat)
        errors.append(dist)
        print(f"   Sample {len(errors):3d}: Pred=({pred_lat:.2f}, {pred_lon:.2f}) "
              f"True=({true_lat:.2f}, {true_lon:.2f}) → {dist:.1f} km")

print(f"\n📊 Errore medio (5 batch): {np.mean(errors):.2f} km")


🌍 TEST ERRORI REALI (KM):
   Sample   1: Pred=(42.08, 12.31) True=(37.51, 15.09) → 560.7 km
   Sample   2: Pred=(44.23, 10.57) True=(45.55, 11.55) → 165.6 km
   Sample   3: Pred=(43.51, 11.42) True=(41.90, 12.48) → 198.8 km
   Sample   4: Pred=(43.99, 11.52) True=(39.22, 9.12) → 566.8 km
   Sample   5: Pred=(42.05, 12.41) True=(41.91, 12.47) → 16.4 km
   Sample   6: Pred=(45.04, 8.22) True=(44.30, 8.48) → 84.5 km
   Sample   7: Pred=(46.08, 10.11) True=(46.51, 10.55) → 57.9 km
   Sample   8: Pred=(43.82, 11.21) True=(38.19, 15.56) → 724.3 km
   Sample   9: Pred=(44.13, 10.48) True=(42.73, 12.74) → 240.0 km
   Sample  10: Pred=(46.74, 8.23) True=(46.66, 8.05) → 16.2 km
   Sample  11: Pred=(43.10, 11.08) True=(42.66, 11.64) → 67.1 km
   Sample  12: Pred=(44.63, 10.89) True=(44.49, 11.34) → 39.3 km
   Sample  13: Pred=(43.78, 11.99) True=(45.23, 11.47) → 166.0 km
   Sample  14: Pred=(46.69, 8.01) True=(46.61, 7.94) → 9.8 km
   Sample  15: Pred=(46.52, 11.02) True=(46.69, 9.43) → 123.3 km

In [11]:
# Image from file — Monte Carlo Dropout inference with heatmap
# Supports both local file paths and URLs.
import os

import folium
import numpy as np
import requests
import torch
from PIL import Image
from folium.plugins import HeatMap
from io import BytesIO
from torchvision import transforms

# --- 1. CONFIGURAZIONE ---
# Devono essere identici a quelli usati nel training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

# --- 2. FUNZIONE PER MC DROPOUT SICURO ---
def enable_dropout(m):
    # Attiva solo il Dropout per l'inferenza Monte Carlo,
    # ma lascia la Batch Normalization in modalità eval (congelata).
    for module in m.modules():
        if module.__class__.__name__.startswith('Dropout'):
            module.train()

# --- 3. GENERATORE DI PROBABILITÀ (SMART) ---
def predict_distribution(model, source, num_samples=50):
    print(f"🔍 Analisi fonte: {source[:60]}...")
    img_pil = None

    # A. GESTIONE URL
    if source.startswith("http"):
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        try:
            response = requests.get(source, headers=headers, timeout=10)
            response.raise_for_status()
            img_pil = Image.open(BytesIO(response.content)).convert("RGB")
        except Exception as e:
            print(f"❌ Errore download URL: {e}")
            return None

    # B. GESTIONE FILE LOCALE
    elif os.path.exists(source):
        try:
            img_pil = Image.open(source).convert("RGB")
        except Exception as e:
            print(f"❌ Errore apertura file locale: {e}")
            return None

    # C. ERRORE
    else:
        print(f"❌ Errore: File non trovato o URL non valido: {source}")
        return None

    # --- INIZIO PREDIZIONE MONTE CARLO ---
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)

    # 1. Metti tutto in eval per bloccare la Batch Norm
    model.eval()
    # 2. Attiva manualmente SOLO il dropout
    enable_dropout(model)

    predictions = []

    with torch.no_grad():
        for _ in range(num_samples):
            # Ogni iterazione il dropout spegnerà neuroni diversi
            preds = model(img_tensor).cpu().numpy()[0]

            # Denormalizzazione
            lat = preds[0] * (MAX_LAT - MIN_LAT) + MIN_LAT
            lon = preds[1] * (MAX_LON - MIN_LON) + MIN_LON
            predictions.append([lat, lon])

    # Rimettiamo il modello standard in eval
    model.eval()

    return np.array(predictions)

# --- 4. CREAZIONE MAPPA HEATMAP ---
def create_geoclip_heatmap(predictions):
    if len(predictions) == 0: return None

    top_n_coordinates = len(predictions)
    gps_coordinates   = predictions.tolist()

    # Probabilità uniforme per la visualizzazione
    probabilities = [1.0 / top_n_coordinates] * top_n_coordinates

    weighted_coordinates = [(lat, lon, weight) for (lat, lon), weight in zip(gps_coordinates, probabilities)]

    # Centro della mappa (Media dei punti)
    avg_lat = sum(lat for lat, lon in gps_coordinates) / len(gps_coordinates)
    avg_lon = sum(lon for lat, lon in gps_coordinates) / len(gps_coordinates)

    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=6, tiles="CartoDB dark_matter")

    # Gradiente Magma
    magma = {
        0.0: '#932667', 0.2: '#b5367a', 0.4: '#d3466b',
        0.6: '#f1605d', 0.8: '#fd9668', 1.0: '#fcfdbf'
    }
    gradient = {str(k): v for k, v in magma.items()}

    HeatMap(weighted_coordinates, radius=25, blur=15, gradient=gradient).add_to(m)

    # Marker sul Centroide
    folium.Marker(
        location=[avg_lat, avg_lon],
        popup=f"Stima Centrale<br>Lat: {avg_lat:.2f}, Lon: {avg_lon:.2f}",
        icon=folium.Icon(color='orange', icon='star')
    ).add_to(m)

    # Pallini ciano per vedere la dispersione (i singoli tentativi)
    for lat, lon in gps_coordinates:
        folium.CircleMarker(
            location=[lat, lon], radius=2, color='cyan', fill=True, fill_opacity=0.3
        ).add_to(m)

    return m

# --- ESEMPIO DI UTILIZZO ---
# Assicurati che 'model' sia caricato in memoria!

# FILE LOCALE (Sostituisci col percorso della tua foto)
local_path = "/content/venezia.jpg"  # <--- CAMBIA QUESTO CON IL TUO FILE
print(f"\n--- TEST LOCALE: {local_path} ---")

from IPython.display import display
if os.path.exists(local_path):
    preds = predict_distribution(model, local_path, num_samples=50)
    if preds is not None:
        mappa = create_geoclip_heatmap(preds)
        display(mappa)
else:
    print(f"⚠️ File non trovato: {local_path}")
    print("   Cambia 'local_path' con il percorso della tua immagine.")

⚠️ Immagine non trovata: /path/to/your/image.jpg
   Modifica la variabile IMAGE_PATH con il percorso corretto.


In [15]:
# REPORT MODELLO — valutazione completa sul validation set
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix


# Funzione Distanza (Haversine — versione numpy per array)
def haversine_np(lon1, lat1, lon2, lat2):
    # Calcola distanza in km tra due array di coordinate
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a    = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c    = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c


def get_macro_region(lat):
    # Assegna una classe (Nord, Centro, Sud) basata sulla latitudine.
    # Semplificazione utile per calcolare Precision/Recall.
    if lat > 44.0:   return "Nord"       # Sopra Bologna
    elif lat > 41.5: return "Centro"     # Tra Roma e Bologna
    else:            return "Sud/Isole"  # Sotto Roma


def evaluate_model_comprehensive(model, val_loader):
    print("📊 Inizio Valutazione Completa (questo richiederà un minuto)...")
    model.eval()

    all_preds_coords = []
    all_true_coords  = []

    # 1. Raccolta Dati
    device_ = next(model.parameters()).device
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs    = imgs.to(device_)
            preds   = model(imgs).cpu().numpy()
            targets = targets.numpy()

            # Denormalizzazione Batch
            for i in range(len(preds)):
                p_lat = preds[i][0]   * (MAX_LAT - MIN_LAT) + MIN_LAT
                p_lon = preds[i][1]   * (MAX_LON - MIN_LON) + MIN_LON
                t_lat = targets[i][0] * (MAX_LAT - MIN_LAT) + MIN_LAT
                t_lon = targets[i][1] * (MAX_LON - MIN_LON) + MIN_LON

                all_preds_coords.append([p_lat, p_lon])
                all_true_coords.append([t_lat, t_lon])

    preds_np = np.array(all_preds_coords)
    true_np  = np.array(all_true_coords)

    # 2. Calcolo Errori in Km
    errors_km = haversine_np(preds_np[:, 1], preds_np[:, 0], true_np[:, 1], true_np[:, 0])

    print("\n--- 📏 STATISTICHE ERRORE (KM) ---")
    print(f"Errore Medio (Mean):   {np.mean(errors_km):.2f} km")
    print(f"Errore Mediano:        {np.median(errors_km):.2f} km")
    print(f"Errore Minimo:         {np.min(errors_km):.2f} km")
    print(f"Errore Massimo:        {np.max(errors_km):.2f} km")

    # 3. Accuracy @ Threshold (La "Recall" geografica)
    print("\n--- 🎯 ACCURACY @ DISTANZA ---")
    for th in [25, 50, 100, 200]:
        acc = np.mean(errors_km <= th) * 100
        print(f"Entro {th} km: {acc:.1f}% delle foto")

    # 4. Precision & Recall per Macro-Regioni
    print("\n--- 🧭 CLASSIFICATION REPORT (NORD/CENTRO/SUD) ---")
    true_regions = [get_macro_region(lat) for lat in true_np[:, 0]]
    pred_regions = [get_macro_region(lat) for lat in preds_np[:, 0]]
    print(classification_report(true_regions, pred_regions, digits=3))

    # 5. Matrice di Confusione Visiva
    labels = ["Nord", "Centro", "Sud/Isole"]
    cm     = confusion_matrix(true_regions, pred_regions, labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predetto dal Modello')
    plt.ylabel('Realtà')
    plt.title('Matrice di Confusione Macro-Regioni')
    plt.show()


# ESECUZIONE
# Assicurati di passare il 'val_loader' che hai definito prima del training
evaluate_model_comprehensive(model, val_loader)

NameError: name 'val_loader' is not defined